# 04 Modelling

This notebook builds the supervised classification stage of the analysis. A
logistic regression and a decision tree are trained to estimate churn risk from
the Trustpilot reviews, using TF-IDF features derived from the processed text.
Both models are evaluated with metrics suited to the class imbalance identified
earlier, and the resulting coefficients and feature importances provide the
ranked churn-risk drivers required by Objective 4. Drivers are then disaggregated
by provider, supporting the dual-use discussion of retention and acquisition
opportunities developed in Chapters 4 and 5.

The data is read from the `reviews_processed` table in the SQLite database,
written at the end of the previous notebook.

In [1]:
import sqlite3
import pandas as pd

connection = sqlite3.connect('telecom_reviews.db')
df = pd.read_sql('SELECT * FROM reviews_processed', connection)
connection.close()

# Dates come back as text from SQLite, as before; convert if needed for any
# time-based work later.
df['date'] = pd.to_datetime(df['date'], errors='coerce')

print(f"Rows read: {len(df)}")
print(f"Columns: {list(df.columns)}")
print()
print(f"Rows with churn_risk label: {df['churn_risk'].notna().sum()}")

Rows read: 1929
Columns: ['provider', 'source', 'rating', 'title', 'text', 'date', 'url', 'text_clean', 'word_count', 'vader_neg', 'vader_neu', 'vader_pos', 'vader_compound', 'text_topics_v1', 'content_word_count', 'text_topics', 'churn_risk', 'dominant_topic_trustpilot', 'dominant_topic_geekzone']

Rows with churn_risk label: 1483


## 1. Feature Engineering

The classification uses two families of features: TF-IDF weights from the
processed text (`text_topics`), and the VADER sentiment scores. The LDA topic
assignment is not included, since it is derived from the same words already
captured by TF-IDF and would add little independent signal, while being harder
to interpret alongside the TF-IDF coefficients. TF-IDF is capped at 300 terms,
which keeps the feature set manageable relative to the number of labelled
reviews and reduces the risk of overfitting.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Take only the labelled Trustpilot reviews.
labelled = df[df['churn_risk'].notna()].copy()
labelled['churn_risk'] = labelled['churn_risk'].astype(int)

print(f"Labelled reviews for modelling: {len(labelled)}")
print()

# Build the TF-IDF matrix from the processed text, same settings as the
# exploratory analysis for consistency, capped at 300 features.
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.5,
    max_features=300
)
X_tfidf = tfidf.fit_transform(labelled['text_topics'])
tfidf_feature_names = tfidf.get_feature_names_out()

print(f"TF-IDF features: {X_tfidf.shape[1]}")

# Add VADER sentiment as extra numeric features: the four VADER scores.
vader_features = labelled[['vader_neg', 'vader_neu', 'vader_pos', 'vader_compound']].values

# Combine TF-IDF (sparse matrix) with the VADER features (dense array).
from scipy.sparse import hstack
X = hstack([X_tfidf, vader_features])

# Build the full list of feature names, in the same order as the columns of X.
feature_names = list(tfidf_feature_names) + ['vader_neg', 'vader_neu', 'vader_pos', 'vader_compound']

y = labelled['churn_risk'].values

print(f"Final feature matrix shape: {X.shape}")
print(f"Total features (TF-IDF + VADER): {len(feature_names)}")

Labelled reviews for modelling: 1483

TF-IDF features: 300
Final feature matrix shape: (1483, 304)
Total features (TF-IDF + VADER): 304


In [3]:
from sklearn.model_selection import train_test_split

# Stratified split: keeps the same class proportion in train and test as in the
# full dataset, essential given the class imbalance.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Training set: {X_train.shape[0]} reviews")
print(f"Test set: {X_test.shape[0]} reviews")
print()

import numpy as np
print("Class distribution in training set:")
print(pd.Series(y_train).value_counts())
print(pd.Series(y_train).value_counts(normalize=True).round(3))
print()
print("Class distribution in test set:")
print(pd.Series(y_test).value_counts())
print(pd.Series(y_test).value_counts(normalize=True).round(3))

Training set: 1186 reviews
Test set: 297 reviews

Class distribution in training set:
1    1064
0     122
Name: count, dtype: int64
1    0.897
0    0.103
Name: proportion, dtype: float64

Class distribution in test set:
1    266
0     31
Name: count, dtype: int64
1    0.896
0    0.104
Name: proportion, dtype: float64


## 2. Logistic Regression

A logistic regression is trained to estimate churn risk from the TF-IDF and
sentiment features. Given the class imbalance identified earlier (89.7% high
risk, 10.3% low risk), class weighting is applied, so that errors on the
minority class are penalised more heavily than errors on the majority class.
This prevents the model from defaulting to the majority class, which would
otherwise achieve high accuracy without learning to distinguish the two groups.

In [4]:
from sklearn.linear_model import LogisticRegression

# class_weight='balanced' automatically weighs each class inversely to its
# frequency, so errors on the rare low-risk class matter more during training.
# max_iter is raised because the default sometimes is not enough for the
# solver to converge with this many features.
logreg = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

logreg.fit(X_train, y_train)

print("Logistic regression trained.")
print(f"Number of features: {logreg.coef_.shape[1]}")

Logistic regression trained.
Number of features: 304


In [5]:
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix, accuracy_score
)

# Get predictions and probability scores on the test set.
y_pred = logreg.predict(X_test)
y_proba = logreg.predict_proba(X_test)[:, 1]  # probability of high risk (class 1)

# Accuracy, shown only to illustrate why it is misleading here.
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.3f}  (misleading here, given the class imbalance)")
print()

# ROC-AUC, the main metric, robust to class imbalance.
auc = roc_auc_score(y_test, y_proba)
print(f"ROC-AUC: {auc:.3f}")
print()

# Precision, recall, F1 for both classes.
print("Classification report:")
print(classification_report(y_test, y_pred, target_names=['Low risk (0)', 'High risk (1)']))

# Confusion matrix: rows are actual classes, columns are predicted classes.
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:")
print(f"                 Predicted Low  Predicted High")
print(f"Actual Low        {cm[0,0]:>10}  {cm[0,1]:>13}")
print(f"Actual High       {cm[1,0]:>10}  {cm[1,1]:>13}")

Accuracy: 0.872  (misleading here, given the class imbalance)

ROC-AUC: 0.957

Classification report:
               precision    recall  f1-score   support

 Low risk (0)       0.44      0.84      0.58        31
High risk (1)       0.98      0.88      0.92       266

     accuracy                           0.87       297
    macro avg       0.71      0.86      0.75       297
 weighted avg       0.92      0.87      0.89       297

Confusion matrix:
                 Predicted Low  Predicted High
Actual Low                26              5
Actual High               33            233


## 3. Decision Tree

A decision tree is trained as a second model, complementing the logistic
regression. While the logistic regression assumes each term contributes
independently to risk, the decision tree can capture interactions between terms
through its branching rules, and its structure is directly interpretable as a
set of if-then conditions. The tree depth is limited to avoid overfitting, and
class weighting is applied for the same reason as in the logistic regression.

In [6]:
from sklearn.tree import DecisionTreeClassifier

# max_depth limits how many questions the tree can ask before deciding, which
# controls overfitting and keeps the tree interpretable.
tree = DecisionTreeClassifier(
    max_depth=5,
    class_weight='balanced',
    random_state=42
)
tree.fit(X_train, y_train)

# Evaluate on the test set, same metrics as the logistic regression.
y_pred_tree = tree.predict(X_test)
y_proba_tree = tree.predict_proba(X_test)[:, 1]

auc_tree = roc_auc_score(y_test, y_proba_tree)
print(f"Decision Tree ROC-AUC: {auc_tree:.3f}")
print()
print("Classification report:")
print(classification_report(y_test, y_pred_tree, target_names=['Low risk (0)', 'High risk (1)']))

cm_tree = confusion_matrix(y_test, y_pred_tree)
print("Confusion matrix:")
print(f"                 Predicted Low  Predicted High")
print(f"Actual Low        {cm_tree[0,0]:>10}  {cm_tree[0,1]:>13}")
print(f"Actual High       {cm_tree[1,0]:>10}  {cm_tree[1,1]:>13}")

Decision Tree ROC-AUC: 0.851

Classification report:
               precision    recall  f1-score   support

 Low risk (0)       0.53      0.77      0.63        31
High risk (1)       0.97      0.92      0.95       266

     accuracy                           0.91       297
    macro avg       0.75      0.85      0.79       297
 weighted avg       0.93      0.91      0.91       297

Confusion matrix:
                 Predicted Low  Predicted High
Actual Low                24              7
Actual High               21            245


In [7]:
# Extract logistic regression coefficients, paired with feature names.
coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': logreg.coef_[0]
})

# Positive coefficients push toward high risk; negative push toward low risk.
print("Top 20 features increasing churn risk (Logistic Regression):")
print(coef_df.sort_values('coefficient', ascending=False).head(20).to_string(index=False))
print()
print("Top 20 features decreasing churn risk (Logistic Regression):")
print(coef_df.sort_values('coefficient', ascending=True).head(20).to_string(index=False))

Top 20 features increasing churn risk (Logistic Regression):
  feature  coefficient
vader_neg     2.053685
  support     1.870498
     hour     1.546417
    month     1.517322
     time     1.482593
    money     1.445702
  account     1.414046
   number     1.287614
   charge     1.268243
   change     1.225701
vader_neu     1.223887
    email     1.109031
     hold     1.058182
      app     1.039555
  payment     1.017869
 provider     0.910439
  useless     0.908703
     chat     0.903120
 contract     0.861663
    leave     0.818000

Top 20 features decreasing churn risk (Logistic Regression):
       feature  coefficient
     vader_pos    -3.252928
          area    -2.389127
         great    -2.364207
     complaint    -2.047822
       helpful    -1.966379
vader_compound    -1.949831
          good    -1.755931
    connection    -1.655967
        system    -1.641695
         happy    -1.537888
     excellent    -1.442124
           man    -1.412498
          team    -1.404226
  

In [8]:
# Extract decision tree feature importances.
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': tree.feature_importances_
})

# Keep only features with non-zero importance (most will be zero, since the
# tree only uses features it actually split on).
importance_df = importance_df[importance_df['importance'] > 0]

print(f"Features used by the tree (non-zero importance): {len(importance_df)}")
print()
print("Top 20 features by importance (Decision Tree):")
print(importance_df.sort_values('importance', ascending=False).head(20).to_string(index=False))

Features used by the tree (non-zero importance): 20

Top 20 features by importance (Decision Tree):
       feature   importance
vader_compound 6.081937e-01
     vader_pos 1.052397e-01
     vader_neg 4.893157e-02
          area 2.910465e-02
     vader_neu 2.816678e-02
           man 2.530251e-02
          kind 2.389987e-02
       process 2.254236e-02
          date 2.025642e-02
     technical 1.499375e-02
          shop 1.482017e-02
     different 1.218616e-02
          long 1.007370e-02
        charge 9.887799e-03
        credit 9.653841e-03
           old 7.036184e-03
      customer 5.608377e-03
         value 4.102427e-03
        people 4.134154e-16
         store 3.841258e-18


## 4. Robustness Check: TF-IDF-Only Models

The decision tree drew nearly 80% of its importance from the VADER scores,
which are strongly correlated with the star rating used to define the target.
To confirm that the word-level drivers hold independently of sentiment, both
models are retrained using TF-IDF features only. This does not replace the
main models, which retain VADER in line with the unified feature matrix
described in Chapter 3; it serves as a robustness check on the service-related
drivers required by Objective 4.

In [9]:
# Rebuild the feature matrix using TF-IDF only, dropping the VADER columns.
X_tfidf_only = X_tfidf  # the TF-IDF matrix built earlier, before hstack with VADER
feature_names_tfidf_only = list(tfidf_feature_names)

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_tfidf_only, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Logistic regression, TF-IDF only.
logreg_t = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
logreg_t.fit(X_train_t, y_train_t)

y_proba_t = logreg_t.predict_proba(X_test_t)[:, 1]
auc_logreg_t = roc_auc_score(y_test_t, y_proba_t)
print(f"Logistic Regression (TF-IDF only) ROC-AUC: {auc_logreg_t:.3f}")

# Decision tree, TF-IDF only.
tree_t = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42)
tree_t.fit(X_train_t, y_train_t)

y_proba_tree_t = tree_t.predict_proba(X_test_t)[:, 1]
auc_tree_t = roc_auc_score(y_test_t, y_proba_tree_t)
print(f"Decision Tree (TF-IDF only) ROC-AUC: {auc_tree_t:.3f}")

Logistic Regression (TF-IDF only) ROC-AUC: 0.935
Decision Tree (TF-IDF only) ROC-AUC: 0.774


In [10]:
# Logistic regression coefficients, TF-IDF only.
coef_df_t = pd.DataFrame({
    'feature': feature_names_tfidf_only,
    'coefficient': logreg_t.coef_[0]
})

print("Top 15 features increasing churn risk (Logistic Regression, TF-IDF only):")
print(coef_df_t.sort_values('coefficient', ascending=False).head(15).to_string(index=False))
print()

# Decision tree importances, TF-IDF only.
importance_df_t = pd.DataFrame({
    'feature': feature_names_tfidf_only,
    'importance': tree_t.feature_importances_
})
importance_df_t = importance_df_t[importance_df_t['importance'] > 0]

print(f"Features used by the tree (TF-IDF only, non-zero importance): {len(importance_df_t)}")
print()
print("Top 15 features by importance (Decision Tree, TF-IDF only):")
print(importance_df_t.sort_values('importance', ascending=False).head(15).to_string(index=False))

Top 15 features increasing churn risk (Logistic Regression, TF-IDF only):
 feature  coefficient
     bad     2.592180
terrible     1.905575
 useless     1.807514
   money     1.797497
    hour     1.685793
  charge     1.614204
   email     1.599176
    rude     1.572817
   month     1.558845
 payment     1.530668
 support     1.482614
 account     1.374995
  cancel     1.318002
horrible     1.274629
  change     1.250988

Features used by the tree (TF-IDF only, non-zero importance): 17

Top 15 features by importance (Decision Tree, TF-IDF only):
       feature   importance
       helpful 2.488987e-01
         great 2.354461e-01
          good 1.800699e-01
      friendly 1.068140e-01
         happy 9.770485e-02
           bad 4.113229e-02
         month 3.167384e-02
        credit 1.484497e-02
        minute 1.178652e-02
          rude 8.022429e-03
          hour 7.801789e-03
        change 7.646428e-03
unprofessional 4.096241e-03
     long time 4.061888e-03
        online 2.542698e-15

In [11]:
# Report how many high-risk and low-risk reviews each provider contributes.
labelled_all = df[df['churn_risk'].notna()].copy()
labelled_all['churn_risk'] = labelled_all['churn_risk'].astype(int)

provider_risk_counts = labelled_all.groupby('provider').agg(
    high_risk_count=('churn_risk', lambda x: (x == 1).sum()),
    low_risk_count=('churn_risk', lambda x: (x == 0).sum())
)
provider_risk_counts['total'] = provider_risk_counts['high_risk_count'] + provider_risk_counts['low_risk_count']

print("High-risk and low-risk review counts by provider:")
print(provider_risk_counts)

High-risk and low-risk review counts by provider:
          high_risk_count  low_risk_count  total
provider                                        
2degrees              540              46    586
One NZ                495              88    583
Spark                 295              19    314


In [12]:
from sklearn.feature_extraction.text import CountVectorizer

def top_high_risk_terms(provider_name, labelled_df, top_n=15):
    """
    For a given provider, compute the terms most distinctive of high-risk
    reviews compared to low-risk reviews, using presence-based proportions,
    the same method used for the overall Trustpilot comparison.
    """
    subset = labelled_df[labelled_df['provider'] == provider_name].copy()

    vec = CountVectorizer(ngram_range=(1, 2), min_df=3, binary=True)
    X_sub = vec.fit_transform(subset['text_topics'])
    terms_sub = vec.get_feature_names_out()

    is_high = (subset['churn_risk'] == 1).values
    is_low = (subset['churn_risk'] == 0).values

    high_prop = np.asarray(X_sub[is_high].mean(axis=0)).ravel()
    low_prop = np.asarray(X_sub[is_low].mean(axis=0)).ravel()

    result = pd.DataFrame({
        'term': terms_sub,
        'high_risk_pct': (high_prop * 100).round(1),
        'low_risk_pct': (low_prop * 100).round(1),
        'difference': ((high_prop - low_prop) * 100).round(1)
    })

    return result.sort_values('difference', ascending=False).head(top_n)


# Run for each of the three providers.
for provider in ['Spark', 'One NZ', '2degrees']:
    n_high = provider_risk_counts.loc[provider, 'high_risk_count']
    n_low = provider_risk_counts.loc[provider, 'low_risk_count']
    print(f"===== {provider} (high risk n={n_high}, low risk n={n_low}) =====")
    print(top_high_risk_terms(provider, labelled_all).to_string(index=False))
    print()

===== Spark (high risk n=295, low risk n=19) =====
            term  high_risk_pct  low_risk_pct  difference
          charge           20.7           0.0        20.7
           month           16.9           0.0        16.9
             day           20.3           5.3        15.1
        provider           14.9           0.0        14.9
             bad           19.7           5.3        14.4
        internet           13.6           0.0        13.6
           money           13.6           0.0        13.6
            year           17.3           5.3        12.0
        terrible           11.9           0.0        11.9
customer service           27.5          15.8        11.7
            bill           11.2           0.0        11.2
          cancel           11.2           0.0        11.2
        customer           42.4          31.6        10.8
          number           10.5           0.0        10.5
         account           14.6           5.3         9.3

===== One NZ (high r

In [13]:
import sqlite3

# 1. Logistic regression drivers, with VADER (the main model).
coef_df_main = coef_df.copy()
coef_df_main['model'] = 'logistic_regression'
coef_df_main['feature_set'] = 'tfidf_plus_vader'

# 2. Decision tree drivers, with VADER (the main model).
importance_df_main = importance_df.copy()
importance_df_main['model'] = 'decision_tree'
importance_df_main['feature_set'] = 'tfidf_plus_vader'
importance_df_main = importance_df_main.rename(columns={'importance': 'coefficient'})

# 3. Logistic regression drivers, TF-IDF only (robustness check).
coef_df_t_copy = coef_df_t.copy()
coef_df_t_copy['model'] = 'logistic_regression'
coef_df_t_copy['feature_set'] = 'tfidf_only'

# 4. Decision tree drivers, TF-IDF only (robustness check).
importance_df_t_copy = importance_df_t.copy()
importance_df_t_copy['model'] = 'decision_tree'
importance_df_t_copy['feature_set'] = 'tfidf_only'
importance_df_t_copy = importance_df_t_copy.rename(columns={'importance': 'coefficient'})

# Combine all four into a single drivers table.
all_drivers = pd.concat(
    [coef_df_main, importance_df_main, coef_df_t_copy, importance_df_t_copy],
    ignore_index=True
)

print(f"Combined drivers table: {len(all_drivers)} rows")
print(all_drivers.head())

Combined drivers table: 641 rows
      feature  coefficient                model       feature_set
0        able    -0.776431  logistic_regression  tfidf_plus_vader
1    absolute    -1.292811  logistic_regression  tfidf_plus_vader
2     account     1.414046  logistic_regression  tfidf_plus_vader
3      action     0.253452  logistic_regression  tfidf_plus_vader
4  additional     0.281625  logistic_regression  tfidf_plus_vader


In [14]:
# Build a single table of provider-level high-risk drivers, from the analysis
# already run, including the sample size for each provider.
provider_driver_rows = []
for provider in ['Spark', 'One NZ', '2degrees']:
    terms = top_high_risk_terms(provider, labelled_all, top_n=15)
    terms['provider'] = provider
    terms['high_risk_n'] = provider_risk_counts.loc[provider, 'high_risk_count']
    terms['low_risk_n'] = provider_risk_counts.loc[provider, 'low_risk_count']
    provider_driver_rows.append(terms)

provider_drivers = pd.concat(provider_driver_rows, ignore_index=True)
print(f"Provider drivers table: {len(provider_drivers)} rows")
print(provider_drivers.head())

Provider drivers table: 45 rows
       term  high_risk_pct  low_risk_pct  difference provider  high_risk_n  \
0    charge           20.7           0.0        20.7    Spark          295   
1     month           16.9           0.0        16.9    Spark          295   
2       day           20.3           5.3        15.1    Spark          295   
3  provider           14.9           0.0        14.9    Spark          295   
4       bad           19.7           5.3        14.4    Spark          295   

   low_risk_n  
0          19  
1          19  
2          19  
3          19  
4          19  


In [15]:
from sklearn.metrics import precision_recall_fscore_support

def get_low_risk_metrics(y_true, y_pred):
    """Extract precision, recall and F1 for the low-risk class (0) automatically."""
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0], zero_division=0
    )
    return precision[0], recall[0], f1[0]

# Main models (with VADER): predictions already computed as y_pred, y_pred_tree.
p_lr, r_lr, f1_lr = get_low_risk_metrics(y_test, y_pred)
p_tree, r_tree, f1_tree = get_low_risk_metrics(y_test, y_pred_tree)

# Robustness models (TF-IDF only): need predictions, not just probabilities.
y_pred_t = logreg_t.predict(X_test_t)
y_pred_tree_t = tree_t.predict(X_test_t)
p_lr_t, r_lr_t, f1_lr_t = get_low_risk_metrics(y_test_t, y_pred_t)
p_tree_t, r_tree_t, f1_tree_t = get_low_risk_metrics(y_test_t, y_pred_tree_t)

# Build the metrics table from these extracted values, not typed manually.
metrics_summary = pd.DataFrame([
    {'model': 'logistic_regression', 'feature_set': 'tfidf_plus_vader',
     'roc_auc': auc, 'low_risk_precision': p_lr, 'low_risk_recall': r_lr, 'low_risk_f1': f1_lr},
    {'model': 'decision_tree', 'feature_set': 'tfidf_plus_vader',
     'roc_auc': auc_tree, 'low_risk_precision': p_tree, 'low_risk_recall': r_tree, 'low_risk_f1': f1_tree},
    {'model': 'logistic_regression', 'feature_set': 'tfidf_only',
     'roc_auc': auc_logreg_t, 'low_risk_precision': p_lr_t, 'low_risk_recall': r_lr_t, 'low_risk_f1': f1_lr_t},
    {'model': 'decision_tree', 'feature_set': 'tfidf_only',
     'roc_auc': auc_tree_t, 'low_risk_precision': p_tree_t, 'low_risk_recall': r_tree_t, 'low_risk_f1': f1_tree_t},
])

metrics_summary[['roc_auc', 'low_risk_precision', 'low_risk_recall', 'low_risk_f1']] = \
    metrics_summary[['roc_auc', 'low_risk_precision', 'low_risk_recall', 'low_risk_f1']].round(3)

print("Metrics extracted automatically:")
print(metrics_summary)
print()

# Save all three tables to the database.
connection = sqlite3.connect('telecom_reviews.db')
all_drivers.to_sql('model_drivers', con=connection, if_exists='replace', index=False)
provider_drivers.to_sql('provider_drivers', con=connection, if_exists='replace', index=False)
metrics_summary.to_sql('model_metrics', con=connection, if_exists='replace', index=False)
connection.commit()
connection.close()

print("Saved three tables to telecom_reviews.db:")
print("  model_drivers    -", len(all_drivers), "rows")
print("  provider_drivers -", len(provider_drivers), "rows")
print("  model_metrics    -", len(metrics_summary), "rows")

Metrics extracted automatically:
                 model       feature_set  roc_auc  low_risk_precision  \
0  logistic_regression  tfidf_plus_vader    0.957               0.441   
1        decision_tree  tfidf_plus_vader    0.851               0.533   
2  logistic_regression        tfidf_only    0.935               0.564   
3        decision_tree        tfidf_only    0.774               0.487   

   low_risk_recall  low_risk_f1  
0            0.839        0.578  
1            0.774        0.632  
2            0.710        0.629  
3            0.613        0.543  

Saved three tables to telecom_reviews.db:
  model_drivers    - 641 rows
  provider_drivers - 45 rows
  model_metrics    - 4 rows


## Confusion matrices for the TF-IDF-only models, for direct comparison with the combined models in Sections 2 and 3.

In [17]:
# Confusion matrices for the TF-IDF-only models, computed and printed the same
# way as the combined models (cells 7 and 9), so Appendix B is fully verifiable
# from saved notebook output rather than reconstructed from Table 4.7.

y_pred_t = logreg_t.predict(X_test_t)
y_pred_tree_t = tree_t.predict(X_test_t)

cm_t = confusion_matrix(y_test_t, y_pred_t)
cm_tree_t = confusion_matrix(y_test_t, y_pred_tree_t)

print("Logistic Regression (TF-IDF only) confusion matrix:")
print(f"                 Predicted Low  Predicted High")
print(f"Actual Low        {cm_t[0,0]:>10}  {cm_t[0,1]:>13}")
print(f"Actual High       {cm_t[1,0]:>10}  {cm_t[1,1]:>13}")
print

print("Decision Tree (TF-IDF only) confusion matrix:")
print(f"                 Predicted Low  Predicted High")
print(f"Actual Low        {cm_tree_t[0,0]:>10}  {cm_tree_t[0,1]:>13}")
print(f"Actual High       {cm_tree_t[1,0]:>10}  {cm_tree_t[1,1]:>13}")

Logistic Regression (TF-IDF only) confusion matrix:
                 Predicted Low  Predicted High
Actual Low                22              9
Actual High               17            249
Decision Tree (TF-IDF only) confusion matrix:
                 Predicted Low  Predicted High
Actual Low                19             12
Actual High               20            246


In [18]:
# Recompute the rating association for 'complaint', 'system' and 'man',
# referenced in Section 4.5 (paragraph on coefficient sign inversion).
tp_rated = df[(df['source'] == 'Trustpilot') & (df['rating'].notna())].copy()
tp_rated['text_topics'] = tp_rated['text_topics'].fillna('').astype(str)

for term in ['complaint', 'system', 'man']:
    has_term = tp_rated[tp_rated['text_topics'].str.split().apply(lambda words: term in words)]
    n = len(has_term)
    one_star = (has_term['rating'] == 1).sum()
    five_star = (has_term['rating'] == 5).sum()
    print(f"'{term}': n={n}, 1-star={one_star} ({one_star/n*100:.1f}%), 5-star={five_star} ({five_star/n*100:.1f}%)")

'complaint': n=37, 1-star=33 (89.2%), 5-star=3 (8.1%)
'system': n=65, 1-star=59 (90.8%), 5-star=3 (4.6%)
'man': n=19, 1-star=10 (52.6%), 5-star=7 (36.8%)


## 5. Quarterly Sentiment Table for Power BI

A quarterly aggregation of review volume and sentiment is built and saved to
the database, to support the Power BI dashboard's sentiment-over-time panel.
The aggregation is broken down by source and provider, so the dashboard can
filter by either.

In [ ]:
# Build a quarterly aggregation of volume and sentiment, broken down by source
# and provider, for the Power BI time-series panel.
df['quarter'] = df['date'].dt.to_period('Q').astype(str)

quarterly_summary = df.groupby(['quarter', 'source', 'provider']).agg(
    review_count=('text', 'count'),
    avg_vader_compound=('vader_compound', 'mean'),
    pct_high_risk=('churn_risk', lambda x: (x == 1).mean() * 100 if x.notna().any() else None)
).reset_index()

quarterly_summary['avg_vader_compound'] = quarterly_summary['avg_vader_compound'].round(3)
quarterly_summary['pct_high_risk'] = quarterly_summary['pct_high_risk'].round(1)

print(f"Quarterly summary table: {len(quarterly_summary)} rows")
print(quarterly_summary.head(10))

# Save to the database.
connection = sqlite3.connect('telecom_reviews.db')
quarterly_summary.to_sql('quarterly_summary', con=connection, if_exists='replace', index=False)
connection.commit()
connection.close()

print()
print("Saved 'quarterly_summary' table to telecom_reviews.db")

In [ ]:
# Connect to the database
conn = sqlite3.connect("telecom_reviews.db")

# See the tables inside the database
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

print(tables)

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("telecom_reviews.db")

tables = ["reviews", "reviews_processed", "model_drivers", "provider_drivers", "model_metrics", "quarterly_summary"]

for table in tables:
    print(f"\n--- {table} ---")
    df_columns = pd.read_sql_query(f"PRAGMA table_info({table});", conn)
    print(df_columns[["name", "type"]])

conn.close()

In [ ]:
conn = sqlite3.connect("telecom_reviews.db")

tables = ["reviews", "reviews_processed", "model_drivers", "provider_drivers", "model_metrics", "quarterly_summary"]

for table in tables:
    df = pd.read_sql_query(f"SELECT * FROM {table};", conn)
    df.to_csv(f"{table}.csv", index=False)

conn.close()

print("CSV files exported.")